# 预训练（Pre-training）与监督微调（SFT）的底层原理

## 预训练的无监督魔法与 Next Token Prediction 损失计算
1. Next Token Prediction（预测下一个词）看似简单，为什么能让大模型获得惊人的智能？
2. 在代码实现中，Logits 和 Labels 之间那个神秘的“错位（Shift）”操作到底是怎样进行的？

#### 预训练的本质——自监督语言建模（CLM）
在传统的深度学习中，我们需要耗费巨大的人工去标注数据（例如给图片贴上“猫”或“狗”的标签）。而大模型之所以能迎来爆发，核心在于它采用了**自监督学习（Self-Supervised Learning）**，其最主流的训练任务被称为**因果语言建模（Causal Language Modeling, CLM）** 。

它的规则极其简单：**大模型通过“预测下一个词”（Next Token Prediction）的机制，在海量无标注文本中自发学习语言规律**。

**为什么“预测下一个词”能带来智能？**
很多人觉得这只是个高级的概率拼接游戏，但当海量语料与庞大的参数量交织时，为了完美预测下一个词，模型必须被迫理解文本背后的逻辑、常识甚至复杂的推理链条：
   * 文本：“中国的首都是____” $\rightarrow$ 模型必须压缩地理和政治常识才能预测出“北京”。
   * 文本：“如果把水加热到100度，它就会____” $\rightarrow$ 模型必须压缩物理规律才能预测出“沸腾”。
   * 文本：“张三比李四高，李四比王五高，所以最矮的是____” $\rightarrow$ 模型必须具备逻辑推理能力才能预测出“王五”。

 因此，“预测下一个词”表面上是统计概率，本质上是大模型在无损压缩人类世界的全部知识。

#### 预训练的工程支柱——交叉熵损失与错位（Shift）
在训练时，为了高效利用 GPU 的大规模并行算力，我们不会一个词一个词地去喂给模型，而是把整句话打包并行的喂进去。

假设我们的训练样本是一句话的 Token ID 序列：["我", "喜欢", "吃", "苹果"]。
我们把这句话同时当做输入（Input）和目标（Label）。但这里有一个关键的工程细节：错位（Shift）。
* 当模型看到 ["我"] 时，期望的输出（Label）是 ["喜欢"]
* 当模型看到 ["我", "喜欢"] 时，期望的输出（Label）是 ["吃"]
* 当模型看到 ["我", "喜欢", "吃"] 时，期望的输出（Label）是 ["苹果"]

在工程实现中，我们让模型对整句话前向传播得到全部位置的 `Logits`，形状为 `(Batch, Seq_Len, Vocab_Size)`。
然后，我们将 `Logits` 的最后一个词时序切掉，将 `Labels` 的第一个词切掉，让它们在时间轴上实现完美的对齐错位：
$$\text{Logits}_{\text{cut}} = \text{Logits}[:, :-1, :] \quad \longleftrightarrow \quad \text{Labels}_{\text{cut}} = \text{Labels}[:, 1:]$$
接着，我们把这两者展平，送入交叉熵损失函数（Cross Entropy Loss）。

交叉熵的物理意义非常直观：如果模型给正确词分配的概率越接近 1，Loss 就越接近 0；如果模型把很高的概率给了错误的词，Loss 就会飙升。大模型的整个预训练阶段，就是通过梯度下降，不断把这个全局的 Cross Entropy Loss 压到最低的过程。

#### 大模型预训练的 Loss 计算与训练步骤
我们使用 PyTorch 模拟大模型在一个极小文本库上的单步预训练过程，重点展示数据是如何被加工、错位，并最终计算出损失的。
| 维度 | 类比 | 含义 |
| :--- | :--- | :--- |
| `batch_size` | 书架上有几排 | 同时处理几个句子 |
| `seq_len` | 每排有几本书 | 每个句子有几个 token |
| `d_model` | 每本书有幾页 | 每个 token 用多少维向量表示 |



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 搭建一个玩具级语言模型
class ToyLM(nn.Module):
    # vocab_size-词表大小能识别的token数量，d_model模型维度
    def __init__(self, vocab_size, d_model):
        super(ToyLM, self).__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        # 简化版：这里只用一个线性层来模拟整个 Transformer 主干网络
        self.backbone = nn.Linear(d_model, d_model)
        # 语言模型头 (LM Head)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    # 前向传播函数 batch_size（批次大小）seq_len（序列长度）d_model（模型维度 / 隐藏层维度）
    def forward(self, idx):
        # idx 形状: (batch_size, seq_len)
        x = self.token_embedding(idx) # 映射为 (batch_size, seq_len, d_model)
        x = F.gelu(self.backbone(x)) # 模拟特征深化激活
        logits = self.lm_head(x) # 映射回词表大小 (batch_size, seq_len, vocab_size)
        return logits

# --- 模拟预训练的一步 (Training Step) ---
if __name__ == '__main__':
    # 固定种子生成固定随机数，只要种子值保持为42，无论在何时何地运行，输出始终一致。
    torch.manual_seed(42)

    # 1. 基础参数定义
    vocab_size = 10 # 假设词表里只有 10 个词 (ID 为 0~9)
    d_model = 16 # 特征维度

    model = ToyLM(vocab_size, d_model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)

    # 2. 模拟一批预训练输入数据 (Batch Size = 2, Sequence Length = 5)
    # 这相当于从互联网上抓取的两句文本对应的 Token ID
    inputs = torch.tensor([
        [1, 3, 5, 7, 2],
        [4, 6, 8, 1, 9]
    ], dtype=torch.long)

    print("【原始输入数据】:\n", inputs, inputs.shape)

    # 3. 前向传播拿到原始 Logits
    logits = model(inputs)
    print("\n 原始 Logits 形状:", logits.shape) # 输出形状: (2, 5, 10)

    # 4. 核心魔法：执行 Shift (错位) 操作
    # logits 扔掉最后一个时序的预测（因为空间上它没有对应的下一个真实词作为 Label 了）
    shift_logits = logits[:, :-1, :].contiguous()
    # labels 扔掉第一个词（因为没有任何前置词可以用来预测第一个词）
    shift_labels = inputs[:, 1:].contiguous()

    print("\n--- Shift 错位变换后 ---")
    print("shift_logits 形状 (Seq_Len 变成了 4):", shift_logits.shape)
    print("shift_labels 形状 (Seq_Len 变成了 4):\n", shift_labels)

    # 5. 展平张量以符合 PyTorch CrossEntropyLoss 的标准输入要求
    # CrossEntropyLoss 期待输入为 (N, C) 和 (N)，其中 N 是所有 Batch 和时序的总和，C 是类别数(Vocab_size)
    flat_logits = shift_logits.view(-1, vocab_size) # 形状变形为: (2*4, 10) -> (8, 10)
    flat_labels = shift_labels.view(-1) # 形状变形为: (2*4) -> (8)

    # 6. 计算因果语言模型损失 (CLM Loss)
    loss = F.cross_entropy(flat_logits, flat_labels)
    print(f"\n当前单步预训练的 Cross Entropy Loss: {loss.item():.4f}")

    # 7. 反向传播与参数更新
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print("参数成功更新】模型向‘更懂人类语言规律’迈进了一小步！")


仔细阅读代码中第 4 步的 `shift_logits` 和 `shift_labels`。如果某一条单独的输入样本是 [A, B, C, D]：
1. shift_logits 代表模型基于哪些输入产生的预测？
2. shift_labels 分别对应哪些真实词？
3. 为什么第 0 个位置的 shift_logits 和第 0 个位置的 shift_labels 能够精准地构成一组“预测”与“标准答案”的映射？

理解 Loss 的物理意义：在刚开始训练时，如果词表大小为 $V$（比如在我们代码里 $V=10$），模型初始化权重完全随机，此时每个词被预测出来的概率都是均等的（即 $1/V$）。根据交叉熵公式 $-\ln(1/V) = \ln(V)$，如果模型完全在瞎猜，初始的 Loss 理论上应该精确地接近多少？请修改代码中的 vocab_size 验证你的推导，并思考为什么在工业界预训练大模型时，通过观察第一个 Step 的 Loss 是不是接近 $\ln(\text{vocab\_size})$ 可以高效用来排查模型网络初始化或数据对齐是否有 Bug？